# Engineering Drawing Review - Demo

Pipeline: **PDF -> PNG image -> Gemini (fixed system prompt) -> structured JSON -> readable report**.

Run the cells top to bottom. To review a different drawing, edit `DRAWING_INDEX` in step 2.

## 0. Setup
The notebook lives in `notebooks/`, so we add the project root to the import path to make `src` importable.

In [ ]:
import sys
from pathlib import Path

# Project root = parent of the notebooks/ folder (no hardcoded paths).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Image, display

from src.pdf_renderer import render_pdf_page
from src.reviewer import list_available_drawings, review_drawing
from src.formatter import format_review
from src.prompts import SYSTEM_PROMPT

## 1. List the available drawings
Every PDF found in `data/easy/` and `data/complex/` is listed with an index.

In [ ]:
drawings = list_available_drawings()
for i, path in enumerate(drawings):
    print(f"[{i}] {path.relative_to(PROJECT_ROOT)}")

## 2. Pick a drawing
Change `DRAWING_INDEX` to switch drawings, and `PAGE` for multi-page PDFs (0 = first page).

In [ ]:
DRAWING_INDEX = 0   # <-- edit this to choose a drawing
PAGE = 0

pdf_path = drawings[DRAWING_INDEX]
print("Selected:", pdf_path.relative_to(PROJECT_ROOT))

## 3. Render the PDF page to an image
PyMuPDF renders the page to a high-resolution PNG (300 DPI by default, see `src/config.py`). The PNG is cached in `outputs/`.

In [ ]:
image_path = render_pdf_page(pdf_path, PAGE)
display(Image(filename=str(image_path), width=900))

## 4. Show the assembled request (dry run)
This prints the model, image info and the full system prompt that will be sent - **without calling the API**.

In [ ]:
_ = review_drawing(pdf_path, page=PAGE, dry_run=True)

## 5. Run the review
Calls Gemini and validates the JSON reply against the schema in `src/schema.py`. Requires a real `GEMINI_API_KEY` in `.env`; with the placeholder key this cell raises a clear error.

In [ ]:
review = review_drawing(pdf_path, page=PAGE)

## 6. Display the report
The formatter always produces the same sections in the same order; missing fields show as "Not specified".

In [ ]:
print(format_review(review))